# Vanilla RAG over the full 10-K corpus (vector store)

Same experiments as `build_agent.ipynb`, scaled up from one NVIDIA filing to **every 10-K for the 10 companies in `data/`**.
Chunks are embedded once and saved to a persistent **Chroma** vector store (`chroma_db/`), so later runs reuse the index instead of re-embedding.

| Experiment | Chunking | Embedding model |
|---|---|---|
| Vanilla RAG 1 | Fixed word windows (150 words, 30 overlap) | `all-MiniLM-L6-v2` |
| Vanilla RAG 2 | Fixed word windows (150 words, 30 overlap) | `BAAI/bge-small-en-v1.5` + query prefix |
| Vanilla RAG 3 | Document structure (Items, headings, tables) | `BAAI/bge-small-en-v1.5` + query prefix |

Numbers are read only from the tables inside the filings. `financials.csv` is **not** used here; it is meant to be added later as a tool (see section 3.4).

**Layout**
1. Setup: imports, config, LLM client
2. Load the corpus
3. Shared building blocks: chunking, vector store, retrieval, answer loop with optional tools
4. Experiments: Vanilla RAG 1, 2, 3
5. Side-by-side comparison

# 1. Setup

In [26]:
import getpass
import json
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import chromadb
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

## Config

Building an index embeds every chunk (about 130k chunks per word-window index for the full corpus), which takes a while on CPU.
It only happens once per experiment; after that the saved collection is reused.
To try things quickly first, set `TICKERS` or `MIN_FISCAL_YEAR` to build a smaller index.

In [27]:
# --- Corpus ---
DATA_DIR = Path("data")
TICKERS = ["NVDA"]            # e.g. ["NVDA", "AAPL"] for a smaller test index; None = all companies
# TICKERS = None    
MIN_FISCAL_YEAR = None    # e.g. 2015 to skip older filings; None = every year

# --- Vector store ---
CHROMA_DIR = "/tmp/chroma_test"  # saved on disk (git-ignored); delete the folder to start from scratch
# CHROMA_DIR = "chroma_db"
REBUILD = False           # True = re-embed even if a complete collection already exists

# --- Retrieval ---
TOP_K = 5
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "  # bge was trained with this on queries

## LLM client

Same as `build_agent.ipynb`: provider from `.env`, API key from the environment or a hidden prompt.

In [28]:
load_dotenv()  # reads LLM_PROVIDER from .env (no keys stored there)

PROVIDER = os.getenv("LLM_PROVIDER", "groq").strip().lower()
if PROVIDER not in ("groq", "openai"):
    raise ValueError(f"LLM_PROVIDER must be 'groq' or 'openai', got '{PROVIDER}'")

# Key comes from the environment if already set this session, otherwise a hidden prompt.
KEY_NAME = "GROQ_API_KEY" if PROVIDER == "groq" else "OPENAI_API_KEY"
if not os.environ.get(KEY_NAME):
    os.environ[KEY_NAME] = getpass.getpass(f"Enter your {KEY_NAME}: ")

if PROVIDER == "groq":
    from groq import Groq
    client = Groq()          # reads GROQ_API_KEY
    MODEL = "openai/gpt-oss-120b"
else:
    from openai import OpenAI
    client = OpenAI()        # reads OPENAI_API_KEY
    MODEL = "YOUR-OPENAI-MODEL"  # fill in a model from platform.openai.com/docs/models

def chat(messages, tools=None, tool_choice="auto", temperature=0.0, max_tokens=1024):
    """Thin wrapper around one chat-completion call. Returns the message object."""
    kwargs = dict(model=MODEL, messages=messages, temperature=temperature, max_tokens=max_tokens)
    if "gpt-oss" in MODEL:
        kwargs["reasoning_effort"] = "low"
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = tool_choice
    return client.chat.completions.create(**kwargs).choices[0].message

# Sanity check (a real API call):
print(f"Provider: {PROVIDER}, model: {MODEL}")
print(chat([{"role": "user", "content": "Reply with exactly one word: ready"}]).content)

Provider: groq, model: openai/gpt-oss-120b
ready


# 2. Load the corpus

`filings_index.csv` lists every 10-K and its folder. Each folder has a readable `10-K.md`, or a `10-K.txt` for 1990s plain-text filings.
The filing's metadata (ticker, fiscal year, URL, ...) is kept with the text so that every chunk can be filtered and cited later.

In [29]:
def load_filings(data_dir=DATA_DIR, tickers=TICKERS, min_year=MIN_FISCAL_YEAR):
    """Return one dict per 10-K: its readable text plus the metadata used for filtering and citations."""
    index = pd.read_csv(data_dir / "filings_index.csv", dtype=str)
    filings = []
    for row in index.itertuples():
        folder = Path(row.local_folder)
        fiscal_year = int(folder.name[2:6])          # folder name is FY{year}_{period-end}
        if (tickers and row.ticker not in tickers) or (min_year and fiscal_year < min_year):
            continue
        path = next((folder / name for name in ("10-K.md", "10-K.txt") if (folder / name).exists()), None)
        if path is None:
            print("Skipped, no readable file:", folder)
            continue
        filings.append({
            "text": path.read_text(encoding="utf-8"),
            "meta": {
                "doc_id": f"{row.ticker}_{folder.name}",
                "ticker": row.ticker,
                "filer": row.filer,
                "fiscal_year": fiscal_year,
                "fiscal_year_end": row.fiscal_year_end,
                "filing_date": row.filing_date,
                "format": path.suffix.lstrip("."),   # "md" or "txt"
                "url": row.url,
            },
        })
    return filings


filings = load_filings()
summary = (pd.DataFrame([{**f["meta"], "words": len(f["text"].split())} for f in filings])
           .groupby("ticker")
           .agg(filings=("doc_id", "count"), first_fy=("fiscal_year", "min"),
                last_fy=("fiscal_year", "max"), words=("words", "sum")))
print(f"{len(filings)} filings, {summary['words'].sum():,} words")
summary

28 filings, 1,476,043 words


,filings,first_fy,last_fy,words
ticker,,,,
NVDA,28,1999,2026,1476043


# 3. Shared building blocks

## 3.1 Chunkers

The same two chunkers as `build_agent.ipynb`. Both return a list of `{"section", "text"}` dicts.

- `chunk_words`: fixed word windows (RAG 1 and 2). It ignores document structure.
- `chunk_by_structure`: splits at 10-K Items, headings and tables (RAG 3), adapted to the Markdown files:
  - headings are `###` or `**bold**` lines; plain-text filings use the short-line heuristic from before
  - tables are runs of `|` rows. They stay whole where possible. A large table is split by rows **with its header repeated**, so every piece still shows which column is which year
  - every chunk stays within `max_words`. bge-small reads at most 512 tokens, and anything longer is silently cut off before embedding

In [30]:
def chunk_words(text, fmt=None, size=150, overlap=30):
    """Fixed-size word windows (Vanilla RAG 1 & 2). `fmt` is unused: windows ignore structure."""
    words = text.split()
    step = size - overlap
    return [{"section": "", "text": " ".join(words[i:i + size])}
            for i in range(0, max(len(words) - overlap, 1), step)]

In [31]:
ITEM_RE = re.compile(r"^item\s+(\d+[a-c]?)\b[.:]?\s*(.*)$", re.IGNORECASE)   # "Item 1A. Risk Factors"
TABLE_SEP_RE = re.compile(r"^\|(\s*:?-{3,}:?\s*\|)+$")                       # "|---|---|"


def n_words(lines):
    return sum(len(line.split()) for line in lines)


def is_heading(line, fmt):
    """Markdown marks headings explicitly; plain-text filings need a heuristic (short, not a sentence)."""
    if fmt == "md":
        return line.startswith("#") or (line.startswith("**") and line.endswith("**") and len(line) < 150)
    words = line.split()
    return (len(words) <= 8 and line[0].isupper()
            and not line.endswith((".", ":", ";", ","))
            and not re.fullmatch(r"[\d\W]+", line))


def split_table(rows, max_words):
    """Split a Markdown table into pieces of whole rows, repeating the header rows in every piece."""
    sep = next((i for i, row in enumerate(rows) if TABLE_SEP_RE.match(row)), None)
    head_end = 0
    if sep is not None:
        head_end = sep + 1
        # Column labels such as years or "(In millions)" often sit just below the separator, with an empty first cell
        while head_end < len(rows) and rows[head_end].split("|")[1].strip() == "":
            head_end += 1
    header, body = rows[:head_end], rows[head_end:]

    pieces, current = [], []
    for row in body:
        if current and n_words(header + current + [row]) > max_words:
            pieces.append(header + current)
            current = []
        current.append(row)
    pieces.append(header + current)
    return ["\n".join(piece) for piece in pieces]


def chunk_by_structure(text, fmt="md", max_words=200):
    """Split at 10-K Items, headings and tables (Vanilla RAG 3). Each chunk starts with its section path."""
    chunks, paragraph, table = [], [], []
    item, item_no, heading = "", "", ""

    def label():
        return " > ".join(part for part in (item, heading) if part)

    def add(body, kind=""):
        chunks.append({"section": label(), "text": f"{label()}{kind}\n{body}".strip()})

    def flush_paragraph():
        if paragraph:
            add(" ".join(paragraph))
            paragraph.clear()

    def flush_table():
        if table:
            for piece in split_table(table, max_words):
                add(piece, " (table)")
            table.clear()

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if line.startswith("|"):                                 # Markdown table row
            flush_paragraph()
            table.append(line)
            continue
        flush_table()
        if not line or line == "Table of Contents" or line.isdigit():
            continue                                             # blank lines and page furniture

        plain = line.strip("#* ")                                # drop Markdown heading / bold markers
        item_match = ITEM_RE.match(plain)
        if item_match and len(plain.split()) <= 20:              # e.g. "Item 1A. Risk Factors" starts a new part
            if item_match.group(1).upper() != item_no:           # repeated page headers like "Item 1A" are ignored
                flush_paragraph()
                item, item_no, heading = plain, item_match.group(1).upper(), ""
        elif is_heading(line, fmt):
            flush_paragraph()
            heading = plain
        else:
            words = plain.split()
            for start in range(0, len(words), max_words):        # very long paragraphs are cut into max_words pieces
                piece = words[start:start + max_words]
                if paragraph and n_words(paragraph) + len(piece) > max_words:
                    flush_paragraph()
                paragraph.append(" ".join(piece))
    flush_paragraph()
    flush_table()
    return chunks

In [32]:
def chunk_corpus(filings, chunker):
    """Chunk every filing. Each chunk keeps its filing's metadata plus its own section and position."""
    records = []
    for f in filings:
        for i, c in enumerate(chunker(f["text"], f["meta"]["format"])):
            records.append({
                "id": f"{f['meta']['doc_id']}:{i}",
                "text": c["text"],
                "meta": {**f["meta"], "section": c["section"][:300], "chunk": i},
            })
    return records

## 3.2 Vector store

Each experiment gets its own Chroma collection, since chunks and embedding models differ.
We embed with `sentence-transformers` ourselves and pass the vectors to Chroma (`embedding_function=None`), so the query side uses exactly the same model and prefix.
Vectors are normalised and the collection uses cosine distance, so `score = 1 - distance` is the same cosine similarity as in `build_agent.ipynb`.

In [33]:
chroma = chromadb.PersistentClient(path=CHROMA_DIR)


@dataclass
class Index:
    """One experiment's vector index: a Chroma collection plus the model (and query prefix) that built it."""
    name: str
    collection: chromadb.Collection
    embedder: SentenceTransformer
    query_prefix: str = ""


def build_index(name, records, embedder, query_prefix="", batch_size=5000):
    """Embed `records` into a Chroma collection. Reuses the saved collection if it is already complete."""
    collection = chroma.get_or_create_collection(name, embedding_function=None, metadata={"hnsw:space": "cosine"})
    if collection.count() == len(records) and not REBUILD:
        print(f"{name}: reusing {collection.count():,} saved vectors")
        return Index(name, collection, embedder, query_prefix)

    # Missing, incomplete or REBUILD: start the collection fresh
    chroma.delete_collection(name)
    collection = chroma.create_collection(name, embedding_function=None, metadata={"hnsw:space": "cosine"})
    vectors = embedder.encode([r["text"] for r in records], batch_size=64,
                              normalize_embeddings=True, show_progress_bar=True)
    for start in range(0, len(records), batch_size):          # Chroma caps how many rows one add() takes
        batch = records[start:start + batch_size]
        collection.add(ids=[r["id"] for r in batch],
                       documents=[r["text"] for r in batch],
                       metadatas=[r["meta"] for r in batch],
                       embeddings=vectors[start:start + batch_size])
    print(f"{name}: indexed {collection.count():,} vectors")
    return Index(name, collection, embedder, query_prefix)

## 3.3 Retrieval

`retrieve` is the same top-k cosine search as before, with optional **metadata filters**.
With 270+ filings, many chunks look alike (risk factors repeat almost word for word every year), so filtering to one company and year is often what makes retrieval work.
Pass no filters to search the whole corpus.

In [34]:
def make_filter(ticker=None, fiscal_year=None):
    """Chroma `where` filter on filing metadata. None means search everything."""
    conditions = []
    if ticker:
        conditions.append({"ticker": ticker})
    if fiscal_year:
        conditions.append({"fiscal_year": fiscal_year})
    if not conditions:
        return None
    return conditions[0] if len(conditions) == 1 else {"$and": conditions}


def retrieve(index, query, k=TOP_K, ticker=None, fiscal_year=None):
    """Top-k chunks by cosine similarity, optionally limited to one company and/or fiscal year."""
    query_vec = index.embedder.encode([index.query_prefix + query], normalize_embeddings=True)
    res = index.collection.query(query_embeddings=query_vec, n_results=k,
                                 where=make_filter(ticker, fiscal_year),
                                 include=["documents", "metadatas", "distances"])
    return [{"id": id_, "score": 1 - dist, "text": doc, "meta": meta}
            for id_, dist, doc, meta in zip(res["ids"][0], res["distances"][0],
                                            res["documents"][0], res["metadatas"][0])]


def show_hits(hits, width=150):
    for n, h in enumerate(hits, 1):               # n matches the [n] citations in the answer
        print(f"[{n}] {h['score']:.3f}  {h['id']}\n       {h['text'][:width]}\n")


def phrase_rank(index, phrase, question, depth=200, **filters):
    """Rank of chunks containing `phrase` for `question`: shows whether the right chunk was found but ranked low."""
    ranked = retrieve(index, question, k=depth, **filters)
    found = [(rank, h) for rank, h in enumerate(ranked, 1) if phrase.lower() in h["text"].lower()]
    if not found:
        print(f"No chunk containing '{phrase}' in the top {depth}")
    for rank, h in found:
        print(f"rank {rank} of top {depth}: {h['id']}\n       {h['text'][:200]}\n")

### Test questions

The same questions as `build_agent.ipynb`. `NVDA_2026` limits retrieval to the filing used there, so results can be compared directly with the single-document notebook.

In [35]:
Q_EXPORT = "What are NVIDIA's main risk factors related to export controls?"
Q_STRATEGY = "key strategies that shape nvidia's overall business approach"
Q_REVENUE = "What was NVIDIA's total revenue for the year ending 25 Jan 2026?"
NVDA_2026 = {"ticker": "NVDA", "fiscal_year": 2026}


def retrieval_checks(index):
    """The retrieval checks every experiment runs, in the same order as build_agent.ipynb."""
    print("=== Export controls, scoped to NVDA FY2026 ===")
    show_hits(retrieve(index, Q_EXPORT, **NVDA_2026))
    print("=== Export controls, whole corpus (does it still find the right filing?) ===")
    show_hits(retrieve(index, Q_EXPORT))
    print("=== Where does the 'key strategies' chunk rank? (NVDA FY2026) ===")
    phrase_rank(index, "key strategies", "What are NVIDIA's key business strategies?", **NVDA_2026)

## 3.4 Answer generation, with room for tools

The vanilla pipeline retrieves once and makes one LLM call. `run_agent` wraps that call in a small loop: if the model asks for a tool, the tool runs, its result goes back to the model, and the model is called again.

**With `TOOLS` empty (as in all experiments here), it is exactly one LLM call**, so results match the plain vanilla setup.
To add a tool later (for example a `financials.csv` lookup), write the function, add its schema to `TOOLS` and register it in `TOOL_FUNCTIONS`. Nothing else changes.

In [36]:
# Tool registry: OpenAI-style function schemas, and the Python function that runs each one.
# Empty for the vanilla experiments.
TOOLS: list[dict] = []
TOOL_FUNCTIONS: dict[str, Callable[..., str]] = {}

# Example of how a tool is added later (not registered):
#
# def query_financials(ticker: str, fiscal_year: int, metric: str) -> str:
#     """Look up one value in data/financials.csv and return it as text."""
#     ...
#
# TOOLS.append({
#     "type": "function",
#     "function": {
#         "name": "query_financials",
#         "description": "Reported annual figure for one company and fiscal year, from XBRL data (FY2009 onwards).",
#         "parameters": {
#             "type": "object",
#             "properties": {
#                 "ticker": {"type": "string", "description": "Stock ticker, e.g. NVDA"},
#                 "fiscal_year": {"type": "integer", "description": "Fiscal year, e.g. 2026"},
#                 "metric": {"type": "string", "enum": ["Revenue", "NetIncome", "OperatingIncome", "EPSDiluted"]},
#             },
#             "required": ["ticker", "fiscal_year", "metric"],
#         },
#     },
# })
# TOOL_FUNCTIONS["query_financials"] = query_financials


def call_tool(call):
    """Run one tool call from the model. Errors go back to the model as text instead of crashing the loop."""
    fn = TOOL_FUNCTIONS.get(call.function.name)
    if fn is None:
        return f"Error: unknown tool '{call.function.name}'"
    try:
        return str(fn(**json.loads(call.function.arguments or "{}")))
    except Exception as e:
        return f"Error running {call.function.name}: {e}"


def run_agent(messages, max_steps=5, max_tokens=1500):
    """Call the model until it gives a final answer, running any tools it asks for along the way."""
    messages = list(messages)
    for _ in range(max_steps):
        msg = chat(messages, tools=TOOLS or None, max_tokens=max_tokens)
        if not msg.tool_calls:
            return msg.content
        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [{"id": c.id, "type": "function",
                            "function": {"name": c.function.name, "arguments": c.function.arguments}}
                           for c in msg.tool_calls],
        })
        for c in msg.tool_calls:
            messages.append({"role": "tool", "tool_call_id": c.id, "content": call_tool(c)})
    return "Stopped: reached the step limit without a final answer."

In [37]:
RAG_SYSTEM = ("Answer the question using ONLY the context below. Cite the chunk ids you used, like [2]. "
              "Each chunk is labelled with its company and fiscal year. "
              "If the answer is not in the context, reply exactly: Not found in the provided context.")


def rag_answer(index, question, k=TOP_K, **filters):
    """Retrieve the top-k chunks, then answer from them only."""
    hits = retrieve(index, question, k, **filters)
    context = "\n\n".join(f"[{n}] ({h['meta']['ticker']} FY{h['meta']['fiscal_year']} 10-K) {h['text']}"
                           for n, h in enumerate(hits, 1))
    answer = run_agent([{"role": "system", "content": RAG_SYSTEM},
                        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])
    return {"answer": answer, "hits": hits}

# 4. Experiments

## Vanilla RAG 1: word windows + `all-MiniLM-L6-v2`

In [38]:
word_records = chunk_corpus(filings, chunk_words)
print(f"{len(word_records):,} chunks")
print(word_records[100]["id"], "\n", word_records[100]["text"][:500])

12,307 chunks
NVDA_FY2026_2026-01-25:100 
 in the future decrease the usage of GPUs for Ethereum mining. This has created and may in the future create increased aftermarket sales of our GPUs, which could negatively impact retail prices for our GPUs and reduce demand for our new GPUs. In general, our new products or previously sold products may be resold online or on the unauthorized “gray market,” which also makes demand forecasting difficult. Gray market products and reseller marketplaces compete with our new products and distribution c


In [39]:
minilm = SentenceTransformer("all-MiniLM-L6-v2")
rag1 = build_index("rag1_words_minilm", word_records, minilm)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

rag1_words_minilm: reusing 12,307 saved vectors


In [40]:
retrieval_checks(rag1)

=== Export controls, scoped to NVDA FY2026 ===
[1] 0.700  NVDA_FY2026_2026-01-25:164
       our business and financial results. Export controls targeting GPUs and semiconductors associated with AI have subjected and may in the future subject 

[2] 0.692  NVDA_FY2026_2026-01-25:181
       our networking products, such as high-speed network interconnects, to limit the ability of downstream parties to create large clusters for frontier mo

[3] 0.643  NVDA_FY2026_2026-01-25:182
       purchase from our competitors rather than NVIDIA or other U.S. firms, harming our business, market position, and financial results. As a result, expor

[4] 0.631  NVDA_FY2026_2026-01-25:180
       restrict additional gaming products, it may disrupt a significant portion of our supply and distribution chain and negatively impact sales of such pro

[5] 0.601  NVDA_FY2026_2026-01-25:165
       of additional export controls has negatively impacted and may in the future negatively impact demand for our products, b

In [41]:
out = rag_answer(rag1, Q_STRATEGY, **NVDA_2026)
# out = rag_answer(rag1, Q_REVENUE, **NVDA_2026)
# out = rag_answer(rag1, Q_STRATEGY)                  # whole corpus, no filter
print(out["answer"])
show_hits(out["hits"])

NVIDIA’s overall business approach is built around a few core strategies that are repeatedly highlighted in its 2026 filing:

* **Pioneering accelerated computing** – The company positions itself as the originator of accelerated‑computing technology and uses that foundation to “solve the most challenging computational problems” across many industries【1】.  
* **Expanding from PC graphics into broader AI‑driven markets** – While its roots are in PC graphics, NVIDIA has deliberately broadened its addressable markets to include scientific computing, AI, data science, autonomous vehicles, robotics, digital twins and other “computationally intensive fields”【2】.  
* **Organizing the business into two operating segments** – All of its products and services are grouped under the **Compute & Networking** segment (data‑center AI platforms, networking, automotive solutions) and the **Graphics** segment (GeForce, RTX/Quadro GPUs). This segmentation focuses resources on the two primary growth engine

## Vanilla RAG 2: word windows + `BAAI/bge-small-en-v1.5`

Same chunks, different embedding model. bge expects `BGE_QUERY_PREFIX` on queries (not on chunks), because of how it was trained.

In [42]:
bge = SentenceTransformer("BAAI/bge-small-en-v1.5")
rag2 = build_index("rag2_words_bge", word_records, bge, query_prefix=BGE_QUERY_PREFIX)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

rag2_words_bge: reusing 12,307 saved vectors


In [43]:
retrieval_checks(rag2)

=== Export controls, scoped to NVDA FY2026 ===
[1] 0.821  NVDA_FY2026_2026-01-25:164
       our business and financial results. Export controls targeting GPUs and semiconductors associated with AI have subjected and may in the future subject 

[2] 0.796  NVDA_FY2026_2026-01-25:181
       our networking products, such as high-speed network interconnects, to limit the ability of downstream parties to create large clusters for frontier mo

[3] 0.753  NVDA_FY2026_2026-01-25:182
       purchase from our competitors rather than NVIDIA or other U.S. firms, harming our business, market position, and financial results. As a result, expor

[4] 0.748  NVDA_FY2026_2026-01-25:180
       restrict additional gaming products, it may disrupt a significant portion of our supply and distribution chain and negatively impact sales of such pro

[5] 0.736  NVDA_FY2026_2026-01-25:162
       for training, modifying, tuning, and deploying LLMs and other AI applications. Such restrictions have limited and could 

In [44]:
out = rag_answer(rag2, Q_STRATEGY, **NVDA_2026)
# out = rag_answer(rag2, Q_REVENUE, **NVDA_2026)
# out = rag_answer(rag2, Q_STRATEGY)
print(out["answer"])
show_hits(out["hits"])

NVIDIA’s overall business approach is guided by a set of core strategies that focus on its accelerated‑computing platform.  The company states that its key strategies include **advancing the NVIDIA accelerated computing platform**—leveraging full‑stack innovation across architecture, chip design, systems, interconnects, algorithms and software to deliver performance leaps that outpace Moore’s Law and provide order‑of‑magnitude advantages for its target markets【1†L1-L5】【4†L1-L5】.
[1] 0.893  NVDA_FY2026_2026-01-25:34
       services, and intelligent in-vehicle experiences. **Business Strategies** NVIDIA’s key strategies that shape our overall business approach include: Ad

[2] 0.803  NVDA_FY2026_2026-01-25:14
       ### Item 1. Business **Our Company** NVIDIA pioneered accelerated computing to help solve the most challenging computational problems. NVIDIA is now a

[3] 0.766  NVDA_FY2026_2026-01-25:229
       as other cautionary statements and risks described elsewhere in this Annual Rep

## Vanilla RAG 3: structure-based chunks + `BAAI/bge-small-en-v1.5`

Same model as RAG 2, but chunks follow the document's structure, so sections and tables are not cut off mid-way.
The chunks change, so this needs **its own collection with fresh embeddings**.

In [45]:
struct_records = chunk_corpus(filings, chunk_by_structure)
print(f"{len(struct_records):,} chunks")
example = next(r for r in struct_records if r["meta"]["doc_id"].startswith("NVDA_FY2026") and "Business Strategies" in r["text"])
print(example["id"], "\n", example["text"][:500])

12,805 chunks
NVDA_FY2026_2026-01-25:32 
 Item 1. Business > Business Strategies
NVIDIA’s key strategies that shape our overall business approach include: Advancing the NVIDIA accelerated computing platform. Our accelerated computing platform can solve complex problems in significantly less time and with lower power consumption than alternative computational approaches. It can help solve problems that were previously deemed unsolvable. We work to deliver continued performance leaps that outpace Moore’s Law by leveraging innovation acros


In [46]:
rag3 = build_index("rag3_structure_bge", struct_records, bge, query_prefix=BGE_QUERY_PREFIX)

rag3_structure_bge: reusing 12,805 saved vectors


In [47]:
retrieval_checks(rag3)

=== Export controls, scoped to NVDA FY2026 ===
[1] 0.882  NVDA_FY2026_2026-01-25:153
       Item 1A. Risk Factors > Risks Related to Regulatory, Legal, Our Stock and Other Matters
Export controls restricting our ability to sell data center GP

[2] 0.872  NVDA_FY2026_2026-01-25:141
       Item 1A. Risk Factors > Risks Related to Regulatory, Legal, Our Stock and Other Matters
and financial results. Export controls targeting GPUs and semi

[3] 0.804  NVDA_FY2026_2026-01-25:140
       Item 1A. Risk Factors > Risks Related to Regulatory, Legal, Our Stock and Other Matters
Such restrictions could include additional unilateral or multi

[4] 0.802  NVDA_FY2026_2026-01-25:152
       Item 1A. Risk Factors > Risks Related to Regulatory, Legal, Our Stock and Other Matters
Open-source foundation models are rapidly growing in popularit

[5] 0.777  NVDA_FY2026_2026-01-25:147
       Item 1A. Risk Factors > Risks Related to Regulatory, Legal, Our Stock and Other Matters
The export controls applicable t

In [48]:
out = rag_answer(rag3, Q_STRATEGY, **NVDA_2026)
# out = rag_answer(rag3, Q_REVENUE, **NVDA_2026)
# out = rag_answer(rag3, Q_STRATEGY)
print(out["answer"])
show_hits(out["hits"])

NVIDIA’s overall business approach is driven by several core strategies:

* **Advance the NVIDIA accelerated‑computing platform** – Delivering full‑stack performance gains that outpace Moore’s Law by innovating across architecture, chip design, systems, interconnects, algorithms and software, and applying this unified, programmable architecture to multiple multi‑billion‑dollar markets such as data‑center, gaming, professional visualization and automotive【1】.  

* **Extend technology and platform leadership in AI** – Provide an end‑to‑end accelerated AI platform for both training and inference, covering GPUs, CPUs, DPUs, networking, systems and software, and expanding AI‑specific GPU features and AI‑focused software suites (e.g., NVIDIA AI Enterprise, NIM, NeMo, AI Blueprints)【3】.  

* **Extend technology and platform leadership in computer graphics** – Fuse AI with graphics to drive growth in consumer entertainment, professional visualization, gaming, design and creative industries, le

# 5. Side-by-side comparison

The top retrieved chunk for each question under each experiment. Add questions to `QUESTIONS` as your evaluation set grows.

Scores are only comparable **within** one embedding model. MiniLM and bge use different similarity scales, so compare which chunk is retrieved, not the raw scores.

In [49]:
QUESTIONS = [
    (Q_EXPORT, NVDA_2026),
    (Q_STRATEGY, NVDA_2026),
    (Q_REVENUE, NVDA_2026),
    (Q_STRATEGY, {}),                                     # same question without filters: right company and year?
]

rows = []
for index in (rag1, rag2, rag3):
    for question, filters in QUESTIONS:
        top = retrieve(index, question, k=1, **filters)[0]
        rows.append({"experiment": index.name, "question": question[:55],
                     "filtered": bool(filters), "score": round(top["score"], 3),
                     "top chunk": top["id"], "preview": top["text"][:120]})

pd.set_option("display.max_colwidth", 120)
pd.DataFrame(rows)

,experiment,question,filtered,score,top chunk,preview
0,rag1_words_minilm,What are NVIDIA's main risk factors related to export c,True,0.700,NVDA_FY2026_2026-01-25:164,our business and financial results. Export controls targeting GPUs and semiconductors associated with AI have subjec...
1,rag1_words_minilm,key strategies that shape nvidia's overall business app,True,0.690,NVDA_FY2026_2026-01-25:13,"“our,” or the “Company” mean NVIDIA Corporation and its subsidiaries. In addition, statements that “we believe” and ..."
2,rag1_words_minilm,What was NVIDIA's total revenue for the year ending 25,True,0.753,NVDA_FY2026_2026-01-25:382,revenue related to hardware and software support and cloud services. **Deferred Revenue** The following table shows ...
3,rag1_words_minilm,key strategies that shape nvidia's overall business app,False,0.738,NVDA_FY2017_2017-01-29:61,be adversely impacted. NVIDIA-branded solutions and services are visual computing and accelerated computing platform...
4,rag2_words_bge,What are NVIDIA's main risk factors related to export c,True,0.821,NVDA_FY2026_2026-01-25:164,our business and financial results. Export controls targeting GPUs and semiconductors associated with AI have subjec...
5,rag2_words_bge,key strategies that shape nvidia's overall business app,True,0.893,NVDA_FY2026_2026-01-25:34,"services, and intelligent in-vehicle experiences. **Business Strategies** NVIDIA’s key strategies that shape our ove..."
6,rag2_words_bge,What was NVIDIA's total revenue for the year ending 25,True,0.792,NVDA_FY2026_2026-01-25:424,"years 2026, 2025, and 2024 was $1.6 billion, $732 million, and $457 million, respectively. Depreciation and amortiza..."
7,rag2_words_bge,key strategies that shape nvidia's overall business app,False,0.893,NVDA_FY2026_2026-01-25:34,"services, and intelligent in-vehicle experiences. **Business Strategies** NVIDIA’s key strategies that shape our ove..."
8,rag3_structure_bge,What are NVIDIA's main risk factors related to export c,True,0.882,NVDA_FY2026_2026-01-25:153,"Item 1A. Risk Factors > Risks Related to Regulatory, Legal, Our Stock and Other Matters\nExport controls restricting..."
9,rag3_structure_bge,key strategies that shape nvidia's overall business app,True,0.902,NVDA_FY2026_2026-01-25:32,Item 1. Business > Business Strategies\nNVIDIA’s key strategies that shape our overall business approach include: Ad...
